In [ ]:
!pip install openpyxl
!pip install imblearn
!pip install opencv-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import RFE, mutual_info_classif

import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.models as models
import torch.optim as optim
from torchvision import transforms

from imblearn.over_sampling import SMOTE

import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
dataset_paths = ["/content/drive/My Drive/pcos_detection/1/content/data/enhanced_data"]  # Change for test set too

corrupt_images = []
for paths in dataset_paths:
  for root, dirs, files in os.walk(paths):
      for file in files:
          img_path = os.path.join(root, file)
          try:
              img = Image.open(img_path).convert("RGB")
          except (UnidentifiedImageError, OSError):
              print(f"⚠️ Corrupt image found: {img_path}")
              corrupt_images.append(img_path)

  print(f"Total corrupt images: {len(corrupt_images)}")

  # 🔹 Optionally delete corrupt images
  for img_path in corrupt_images:
      os.remove(img_path)
      print(f"🗑️ Deleted: {img_path}")


Total corrupt images: 0


In [ ]:
import os
from PIL import Image
import numpy as np
import cv2  # OpenCV for image processing
from torch.utils.data import Dataset
from torchvision import transforms

class PCOSImageDataset(Dataset):
    def __init__(self, image_folder, transform=None, augment=True):
        self.image_paths = []
        self.labels = []
        class_mapping = {"notinfected": 0, "infected": 1}

        for class_name in os.listdir(image_folder):
            class_path = os.path.join(image_folder, class_name)
            if os.path.isdir(class_path):  # Ensure it's a directory
                for img_name in os.listdir(class_path):
                    img_path = os.path.join(class_path, img_name)
                    # Only load valid image formats
                    if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append(img_path)
                        self.labels.append(class_mapping[class_name])  # Assign label based on folder name

        # Data Augmentation transforms (if augment is True)
        self.augment = augment

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),  # Resize images to 224x224
            transforms.ToTensor(),  # Convert images to tensor
        ]) if transform is None else transform  # Use provided transform if any

        # Augmentation transformations
        self.augmentation_transform = transforms.Compose([
            transforms.RandomRotation(degrees=30),  # Random Rotation (-30 to 30 degrees)
            transforms.RandomHorizontalFlip(),  # Random horizontal flip
            transforms.RandomVerticalFlip(),  # Random vertical flip
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # Zooming, crop with random scaling
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),  # Brightness adjustments
        ]) if augment else None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")  # Convert image to RGB

        # Convert to numpy array for processing
        image_np = np.array(image)

        # Apply Watershed Segmentation
        image_segmented = self.apply_watershed(image_np)

        # Apply Multilevel Thresholding to detect cysts
        image_segmented = self.apply_multilevel_thresholding(image_segmented)

        # Convert back to PIL Image
        image_segmented = Image.fromarray(image_segmented)

        # Apply Augmentation if enabled
        if self.augment:
            image_segmented = self.augmentation_transform(image_segmented)  # Apply augmentation

        # Apply basic transformations (resize + tensor conversion)
        image_segmented = self.transform(image_segmented)

        label = self.labels[idx]
        return image_segmented, label

    def apply_watershed(self, image):
        """
        Apply Watershed Segmentation to the image.
        This method detects the follicle boundaries based on intensity changes.
        """
        # Step 1: Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        # Step 2: Apply thresholding to get a binary image (foreground vs background)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Step 3: Remove noise using morphological operations
        kernel = np.ones((3, 3), np.uint8)
        sure_bg = cv2.dilate(thresh, kernel, iterations=3)

        # Step 4: Apply distance transform
        dist_transform = cv2.distanceTransform(thresh, cv2.DIST_L2, 5)
        _, sure_fg = cv2.threshold(dist_transform, 0.7 * dist_transform.max(), 255, 0)

        # Step 5: Subtract sure foreground from sure background to get unknown region
        sure_fg = np.uint8(sure_fg)
        unknown = cv2.subtract(sure_bg, sure_fg)

        # Step 6: Label markers (foreground vs background)
        _, markers = cv2.connectedComponents(sure_fg)

        # Step 7: Apply watershed algorithm
        markers = markers + 1
        markers[unknown == 255] = 0

        # Step 8: Perform watershed algorithm
        image_segmented = image.copy()
        cv2.watershed(image_segmented, markers)

        # Mark boundary pixels
        image_segmented[markers == -1] = [255, 0, 0]  # Red boundary lines

        return image_segmented

    def apply_multilevel_thresholding(self, image):
        """
        Apply Multilevel Thresholding to detect cysts.
        This method identifies cysts by using multiple thresholds.
        """
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        # Use Otsu's method for automatic thresholding, or you can manually choose threshold values
        _, threshold_1 = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        _, threshold_2 = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

        # Combine multiple thresholds to create segmented regions
        segmented_image = np.zeros_like(gray)
        segmented_image[threshold_1 == 255] = 1  # Cyst regions marked as 1
        segmented_image[threshold_2 == 255] = 2  # Follicle regions marked as 2

        # Convert back to RGB for visualization
        segmented_image_rgb = cv2.applyColorMap(segmented_image * 85, cv2.COLORMAP_JET)  # Colorize the output
        return segmented_image_rgb


In [ ]:
import torch
from torch.utils.data import random_split, DataLoader  # Fix import

image_dataset = PCOSImageDataset(dataset_paths[0])

# Define split ratios (e.g., 70% train, 15% val, 15% test)
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# Get total dataset size
total_size = len(image_dataset)

# Compute split sizes
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size  # Ensure all samples are used

# Split dataset
train_dataset, val_dataset, test_dataset = random_split(image_dataset, [train_size, val_size, test_size])

# Print dataset sizes
print(f"Train size: {train_size}, Validation size: {val_size}, Test size: {test_size}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


Train size: 10774, Validation size: 2308, Test size: 2310


In [ ]:
# 🔹 Verify the DataLoader Outputs Correct Images for CNN
for images, labels in train_loader:
    print(f"Batch Train Image Shape: {images.shape}")  # Expected: [32, 3, 224, 224]
    print(f"Batch Train Label Shape: {labels.shape}")  # Expected: [32]
    break  # Only check one batch

for images, labels in test_loader:
    print(f"Batch Test Image Shape: {images.shape}")  # Expected: [32, 3, 224, 224]
    print(f"Batch Test Label Shape: {labels.shape}")  # Expected: [32]
    break  # Only check one batch

for images, labels in val_loader:
    print(f"Batch Test Image Shape: {images.shape}")  # Expected: [32, 3, 224, 224]
    print(f"Batch Test Label Shape: {labels.shape}")  # Expected: [32]
    break  # Only check one batch

Batch Train Image Shape: torch.Size([32, 3, 224, 224])
Batch Train Label Shape: torch.Size([32])
Batch Test Image Shape: torch.Size([32, 3, 224, 224])
Batch Test Label Shape: torch.Size([32])
Batch Test Image Shape: torch.Size([32, 3, 224, 224])
Batch Test Label Shape: torch.Size([32])


In [ ]:
# 🔹 CNN Model for Ultrasound Images
class PCOS_CNN(nn.Module):
    def __init__(self, num_classes=2):
        super(PCOS_CNN, self).__init__()
        self.cnn = models.resnet18(pretrained=True)  # Use ResNet18 as the backbone
        self.cnn.fc = nn.Linear(512, 32)  # Modify the last layer to output a 32D feature vector

        self.classifier = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)  # Final classification layer
        )

    def forward(self, x):
        x = self.cnn(x)  # Extract image features
        x = self.classifier(x)  # Pass through the classifier
        return x

# 🔹 Define Model Parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2  # Binary classification (PCOS or not)

# Initialize Model
cnn_model = PCOS_CNN(num_classes).to(device)
print("✅ CNN Model for Ultrasound Images Ready! 🚀")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 262MB/s]


✅ CNN Model for Ultrasound Images Ready! 🚀


In [ ]:
# 🔹 Define Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# 🔹 Updated Training Function with Validation
def train_cnn_model(model, train_loader, val_loader, test_loader, criterion, optimizer, num_epochs=5):
    model.to(device)

    for epoch in range(num_epochs):
        # 🔹 Training Phase
        model.train()
        total_train_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * images.size(0)  # Sum loss for batch
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

        train_loss = total_train_loss / total_train  # Compute average loss
        train_accuracy = 100 * correct_train / total_train

        # 🔹 Validation Phase
        model.eval()
        total_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                total_val_loss += loss.item() * images.size(0)  # Sum loss for batch
                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

        val_loss = total_val_loss / total_val  # Compute average loss
        val_accuracy = 100 * correct_val / total_val


        # Update learning rate
        scheduler.step()

        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%, "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.2f}%")

    save_path = "/content/drive/My Drive/pcos_detection/cnn_model.pth"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")

# 🔹 Train the CNN Model with Validation
train_cnn_model(cnn_model, train_loader, val_loader, test_loader, criterion, optimizer, num_epochs=5)

Epoch [1/5], Train Loss: 0.0357, Train Acc: 98.79%, Val Loss: 0.0205, Val Acc: 99.44%, Test Loss: 0.0027, Test Acc: 99.91%
Epoch [2/5], Train Loss: 0.0165, Train Acc: 99.50%, Val Loss: 0.0128, Val Acc: 99.57%, Test Loss: 0.0027, Test Acc: 99.91%
Epoch [3/5], Train Loss: 0.0146, Train Acc: 99.55%, Val Loss: 0.0115, Val Acc: 99.57%, Test Loss: 0.0027, Test Acc: 99.91%
Epoch [4/5], Train Loss: 0.0143, Train Acc: 99.55%, Val Loss: 0.0041, Val Acc: 99.87%, Test Loss: 0.0027, Test Acc: 99.91%
Epoch [5/5], Train Loss: 0.0195, Train Acc: 99.40%, Val Loss: 0.0184, Val Acc: 99.48%, Test Loss: 0.0027, Test Acc: 99.91%
Model saved to /content/drive/My Drive/pcos_detection/cnn_model.pth


In [ ]:
cnn_model.eval()

total_test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)

        total_test_loss += loss.item() * images.size(0)  # Sum loss for batch
        _, predicted = torch.max(outputs, 1)
        correct_test += (predicted == labels).sum().item()
        total_test += labels.size(0)

test_loss = total_test_loss / total_test  # Compute average loss
test_accuracy = 100 * correct_test / total_test

print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.2f}%")

Test Loss: 0.0129, Test Acc: 99.61%
